# Slow-mo grokking — AdamW per-step training

This notebook trains a 2-block transformer on mod-97 division with **AdamW (LR=1e-3, WD=1.0)**, saves a state_dict at *every* gradient step, and then analyzes how the dlog-Fourier algorithm forms. The output of this notebook produces the per-step data used in Fig. 1 of the article.

**Standalone.** No external data required. Trains from scratch in ~10-12 min on a single GPU. All outputs go to `./outputs/slowmo/` relative to this notebook.

**Setup.** Same seed (1000), same model, same task, same vanilla random init. Optimizer: **AdamW LR=1e-3, WD=1.0, betas=(0.9, 0.98)**, batch 512, 1500 steps.

**Outputs:**
- `snapshots.pt` — per-step state_dicts (large: ~8.7 GB if you save all 911 of them; comment out if you don't want this)
- `summary.json` — per-step accuracy, n_clean Fourier neurons, max purity, W_U circle CV
- `formation_curves.png`, `formation_combined.png`, `frequency_emergence.png`, `purity_evolution.png`

In [ ]:
# Cell 1 — Setup
# If running in Colab and you want outputs on Drive, uncomment the two lines below:
#   from google.colab import drive
#   drive.mount('/content/drive')
# Otherwise outputs are written to ./outputs/slowmo/ next to this notebook.

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, json, time, os, gc
from pathlib import Path
from collections import Counter, defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

OUT_DIR = Path('./outputs/slowmo')
OUT_DIR.mkdir(parents=True, exist_ok=True)

P = 97; PM1 = 96
D_MODEL = 384; D_FFN = 4*D_MODEL; N_HEADS = 12
BATCH = 512
SEED = 1000
MAX_STEPS = 1500

import matplotlib.pyplot as plt
BG, FG, GRID, MUTED, ACC1, ACC2 = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E', '#D45A2A', '#3A6E8C'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.6,
    'axes.labelcolor': FG, 'axes.titlecolor': FG,
    'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.8,
})

def is_gen(g, p):
    seen = set(); x = 1
    for _ in range(p-1):
        x = (x*g) % p
        if x in seen: return False
        seen.add(x)
    return len(seen) == p-1
for g in range(2, P):
    if is_gen(g, P): break
exp_table = np.zeros(PM1, dtype=np.int64); x = 1
for t in range(PM1): exp_table[t] = x; x = (x*g) % P
dlog = np.zeros(P, dtype=np.int64)
for t in range(PM1): dlog[exp_table[t]] = t

print(f'Device: {device}, g={g}, OUT={OUT_DIR.resolve()}')

In [ ]:
# Cell 2 — Model + data
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.scale

class GrokBlock(nn.Module):
    def __init__(self, d, nh, dropout=0.2):
        super().__init__()
        self.norm1 = RMSNorm(d); self.attn = nn.MultiheadAttention(d, nh, dropout=dropout, batch_first=True)
        self.norm2 = RMSNorm(d); self.w1 = nn.Linear(d, 4*d, bias=False)
        self.w2 = nn.Linear(4*d, d, bias=False); self.w3 = nn.Linear(d, 4*d, bias=False)
        self.dropout = nn.Dropout(dropout)
        self._ffn_mid = None
    def forward(self, x):
        h = self.norm1(x); o, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(o)
        h2 = self.norm2(x); gate = F.silu(self.w1(h2))
        ffn_mid = gate * self.w3(h2)
        self._ffn_mid = ffn_mid[:, -1, :].detach()
        return x + self.dropout(self.w2(ffn_mid))

class GrokModel(nn.Module):
    def __init__(self, p=97, d=384, nh=12):
        super().__init__()
        self.tok_emb = nn.Embedding(p+2, d); self.pos_emb = nn.Embedding(4, d)
        self.blocks = nn.ModuleList([GrokBlock(d, nh) for _ in range(2)])
        self.norm_f = RMSNorm(d); self.head = nn.Linear(d, p, bias=False); self.p = p
    def forward(self, a, b):
        B = a.size(0)
        op = torch.full((B,), self.p, device=a.device, dtype=torch.long)
        eq = torch.full((B,), self.p+1, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        for bl in self.blocks: x = bl(x)
        return self.head(self.norm_f(x)[:, -1, :])

all_a, all_b, all_c = [], [], []
for a in range(P):
    for b in range(1, P):
        all_a.append(a); all_b.append(b); all_c.append((a * pow(b, P-2, P)) % P)
all_a = torch.tensor(all_a, device=device)
all_b = torch.tensor(all_b, device=device)
all_c = torch.tensor(all_c, device=device)
g_split = torch.Generator(device=device); g_split.manual_seed(0)
perm_split = torch.randperm(len(all_a), device=device, generator=g_split)
n_train = len(all_a) // 2
train_a, train_b, train_c = all_a[perm_split[:n_train]], all_b[perm_split[:n_train]], all_c[perm_split[:n_train]]
test_a,  test_b,  test_c  = all_a[perm_split[n_train:]], all_b[perm_split[n_train:]], all_c[perm_split[n_train:]]

diff_arr = np.array([
    -1 if int(all_a[i].item()) == 0 else (int(dlog[int(all_a[i].item())]) - int(dlog[int(all_b[i].item())])) % PM1
    for i in range(len(all_a))
])
print(f'Data: {len(train_a)} train, {len(test_a)} test')

In [ ]:
# Cell 3 — AdamW slow-mo training with per-step snapshots (fp16)

torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
m = GrokModel(P, D_MODEL, N_HEADS).to(device)
opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1.0, betas=(0.9, 0.98))
pg = torch.Generator(device=device); pg.manual_seed(SEED + 7)

@torch.no_grad()
def eval_split(model, a, b, c):
    model.eval()
    return (model(a, b).argmax(-1) == c).float().mean().item()

def snapshot_fp16(model, step_label):
    sd = {n: p.data.to(torch.float16).cpu().clone() for n, p in model.named_parameters()}
    test_acc = eval_split(model, test_a, test_b, test_c)
    train_acc = eval_split(model, train_a, train_b, train_c)
    return {'step': step_label, 'state_dict': sd, 'test_acc': test_acc, 'train_acc': train_acc}

snapshots = [snapshot_fp16(m, 'pre')]
print(f'pre: train={snapshots[-1]["train_acc"]:.3f}, test={snapshots[-1]["test_acc"]:.3f}')

step = 0
grok_step = None
t0 = time.time()
while step < MAX_STEPS:
    m.train()
    perm = torch.randperm(len(train_a), device=device, generator=pg)
    for i in range(0, len(train_a), BATCH):
        step += 1
        if step <= 10:
            for grp in opt.param_groups: grp['lr'] = 1e-3 * step / 10
        idx = perm[i:i+BATCH]
        loss = F.cross_entropy(m(train_a[idx], train_b[idx]), train_c[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        snapshots.append(snapshot_fp16(m, step))
        if grok_step is None and snapshots[-1]['test_acc'] >= 0.95:
            grok_step = step
            print(f'>>> GROKKED at step {step}: train={snapshots[-1]["train_acc"]:.3f}, test={snapshots[-1]["test_acc"]:.3f}')
        if step % 100 == 0 or step <= 5:
            s = snapshots[-1]
            print(f'step {step:>4}: train={s["train_acc"]:.3f}, test={s["test_acc"]:.3f}')
        if step >= MAX_STEPS: break
    if grok_step is not None and step > grok_step + 150:
        print(f'Stable post-grok, stopping at step {step}')
        break

print(f'\nTraining done in {time.time()-t0:.1f}s, {len(snapshots)} snapshots')

out_path = OUT_DIR / 'snapshots.pt'
torch.save(snapshots, out_path)
print(f'Saved to {out_path} ({out_path.stat().st_size / 1e9:.1f} GB)')

# Keep snapshots in memory for analysis below (don't delete)

In [ ]:
# Cell 4 — Fourier analysis utilities + per-step processing

def fourier_decompose_dlog(M):
    p = M.shape[0]; K = p//2 + 1
    i = np.arange(p)
    A = np.zeros((K, M.shape[1])); B = np.zeros((K, M.shape[1]))
    for k in range(K):
        c = np.cos(2*np.pi*k*i/p); s = np.sin(2*np.pi*k*i/p)
        norm = p/2 if k != 0 and k != p//2 else p
        A[k] = c @ M / norm; B[k] = s @ M / norm
    return A, B, (A**2).sum(1) + (B**2).sum(1)

def project_2d(M, a, b):
    u = a / (np.linalg.norm(a) + 1e-12)
    v = b - (b @ u) * u; v = v / (np.linalg.norm(v) + 1e-12)
    return np.stack([M @ u, M @ v], axis=1)

def cv_radii(xy):
    cx, cy = xy.mean(0)
    r = np.sqrt((xy[:,0]-cx)**2 + (xy[:,1]-cy)**2)
    return r.std() / (r.mean() + 1e-12)

def neuron_purity(M):
    F_ = np.fft.fft(M, axis=0); pw = (F_.conj()*F_).real
    pw[0] = 0
    total = pw.sum(axis=0) + 1e-12
    dom = np.argmax(pw, axis=0)
    pur = pw[dom, np.arange(M.shape[1])] / total
    dom = np.where(dom > PM1//2, dom - PM1, dom)
    return dom, pur

@torch.no_grad()
def collect_ffn1_act_per_diff(state_dict):
    m_ = GrokModel(P, D_MODEL, N_HEADS).to(device)
    m_.load_state_dict({k: v.float().to(device) for k, v in state_dict.items()}, strict=False); m_.eval()
    out = np.zeros((PM1, D_FFN)); counts = np.zeros(PM1)
    BS = 1024
    for i in range(0, len(all_a), BS):
        ab = all_a[i:i+BS]; bb = all_b[i:i+BS]
        _ = m_(ab, bb)
        mid = m_.blocks[1]._ffn_mid.cpu().numpy()
        for j in range(len(ab)):
            di = diff_arr[i + j]
            if di < 0: continue
            out[di] += mid[j]; counts[di] += 1
    out = out / np.maximum(counts[:, None], 1)
    with torch.no_grad():
        acc = (m_(test_a, test_b).argmax(-1) == test_c).float().mean().item()
    del m_; gc.collect(); torch.cuda.empty_cache()
    return out, acc

results = []
t0 = time.time()
for idx, snap in enumerate(snapshots):
    sd = snap['state_dict']
    act, acc = collect_ffn1_act_per_diff(sd)
    dom, pur = neuron_purity(act)
    n_clean = int((pur > 0.3).sum())
    n_very_clean = int((pur > 0.5).sum())
    max_pur = float(pur.max())
    wu = sd['head.weight'].float().cpu().numpy()
    wu_dlog = wu[exp_table]
    A, B, power = fourier_decompose_dlog(wu_dlog)
    p_no_dc = power.copy(); p_no_dc[0] = 0
    top_freqs = np.argsort(p_no_dc)[::-1][:5].tolist()
    cvs = []
    for k in top_freqs:
        xy = project_2d(wu_dlog, A[k], B[k])
        cvs.append(float(cv_radii(xy)))
    results.append({
        'step': snap['step'], 'test_acc': acc,
        'n_clean': n_clean, 'n_very_clean': n_very_clean, 'max_purity': max_pur,
        'wu_top_freqs': top_freqs, 'wu_best_cv': float(min(cvs)),
        'purity_array': pur.tolist(), 'dom_array': dom.tolist(),
    })
    if idx % 50 == 0 or idx == len(snapshots) - 1:
        elapsed = time.time() - t0
        print(f'  {snap["step"]:>4}: acc={acc:.3f}, n_clean={n_clean}, max_pur={max_pur:.3f}, WU CV={min(cvs):.3f}  [{elapsed:.0f}s]')
print(f'\nAnalyzed {len(results)} steps in {time.time()-t0:.1f}s')

In [ ]:
# Cell 5 — Formation curves (4 panels)
step_num = np.array([-1 if r['step'] == 'pre' else int(r['step']) for r in results])
accs = [r['test_acc'] for r in results]
n_clean = [r['n_clean'] for r in results]
n_very_clean = [r['n_very_clean'] for r in results]
max_pur = [r['max_purity'] for r in results]
wu_cv = [r['wu_best_cv'] for r in results]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ax = axes[0, 0]
ax.plot(step_num, accs, '-', color=ACC1, lw=1.6)
ax.set_xlabel('step'); ax.set_ylabel('test acc'); ax.set_title('Test accuracy (AdamW)')
ax.set_ylim(-0.02, 1.02); ax.axhline(0.95, color=MUTED, ls=':', lw=0.6, alpha=0.5)
ax.axhline(0.50, color='#3A8C3A', ls=':', lw=0.6, alpha=0.5)

ax = axes[0, 1]
ax.plot(step_num, n_clean, '-', color=ACC1, lw=1.6, label='purity > 0.3')
ax.plot(step_num, n_very_clean, '-', color=ACC2, lw=1.4, label='purity > 0.5')
ax.set_xlabel('step'); ax.set_ylabel('# clean FFN1 neurons')
ax.set_title('Fourier filter emergence'); ax.legend(fontsize=8, frameon=False)

ax = axes[1, 0]
ax.plot(step_num, max_pur, '-', color='#3A8C3A', lw=1.6)
ax.set_xlabel('step'); ax.set_ylabel('max purity'); ax.set_title('Sharpest filter'); ax.set_ylim(0, 1.02)

ax = axes[1, 1]
ax.plot(step_num, wu_cv, '-', color='#3A6E8C', lw=1.6)
ax.axhline(0.15, color='#3A8C3A', ls='--', lw=0.6, alpha=0.6)
ax.set_xlabel('step'); ax.set_ylabel('W_U best CV'); ax.set_title('W_U Fourier circle quality')

plt.tight_layout()
plt.savefig(OUT_DIR / 'formation_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6 — Headline combined plot: acc + n_clean + max_purity overlaid
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(step_num, accs, '-', color=ACC1, lw=2, label='test accuracy', zorder=3)
ax1.set_xlabel('gradient step'); ax1.set_ylabel('test accuracy', color=ACC1)
ax1.tick_params(axis='y', labelcolor=ACC1)
ax1.set_ylim(-0.02, 1.05)
ax1.axhline(0.95, color=MUTED, ls=':', lw=0.6, alpha=0.5)
ax1.axhline(0.50, color='#3A8C3A', ls=':', lw=0.6, alpha=0.5)

ax2 = ax1.twinx()
ax2.plot(step_num, n_clean, '-', color=ACC2, lw=1.6, label='# clean FFN1 (purity > 0.3)', alpha=0.85)
ax2.plot(step_num, [m_ * max(n_clean) for m_ in max_pur], '--', color='#3A8C3A',
         lw=1.0, label='max purity (rescaled)', alpha=0.6)
ax2.set_ylabel('# clean Fourier neurons', color=ACC2)
ax2.tick_params(axis='y', labelcolor=ACC2)
ax2.spines['top'].set_visible(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8, frameon=False)
ax1.set_title('Slow-mo grokking AdamW: Fourier neurons vs test accuracy')
plt.tight_layout()
plt.savefig(OUT_DIR / 'formation_combined.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7 — Frequency emergence heatmap
K_MAX = PM1 // 2 + 1
heat = np.zeros((len(results), K_MAX))
for i, r in enumerate(results):
    pur = np.array(r['purity_array'])
    dom = np.abs(np.array(r['dom_array']))
    clean_mask = pur > 0.3
    for k in range(K_MAX):
        heat[i, k] = ((dom == k) & clean_mask).sum()

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(heat.T, aspect='auto', cmap='magma', origin='lower',
               extent=[step_num[0]-0.5, step_num[-1]+0.5, -0.5, K_MAX-0.5])
ax.set_xlabel('gradient step')
ax.set_ylabel('|dominant freq k|')
ax.set_title('AdamW: clean Fourier neurons per frequency, over training')
plt.colorbar(im, ax=ax, label='# neurons (purity > 0.3)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'frequency_emergence.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8 — Purity distribution evolution
N = len(results)
step_picks = sorted(set([0, N//8, N//4, 3*N//8, N//2, 5*N//8, 3*N//4, 7*N//8, N-1]))
fig, axes = plt.subplots(1, len(step_picks), figsize=(1.3 * len(step_picks), 2.8), sharey=True)
bins = np.linspace(0, 1, 50)
for ax, idx in zip(axes, step_picks):
    pur = np.array(results[idx]['purity_array'])
    ax.hist(pur, bins=bins, color=ACC1, alpha=0.85, edgecolor=BG, linewidth=0.3)
    ax.axvline(0.3, color=MUTED, ls=':', lw=0.6, alpha=0.5)
    ax.axvline(0.5, color=MUTED, ls=':', lw=0.6, alpha=0.5)
    s = results[idx]['step']; a = results[idx]['test_acc']
    ax.set_title(f'step {s}\nacc={a:.2f}', fontsize=9)
    ax.set_xlabel('purity', fontsize=8); ax.tick_params(labelsize=7)
axes[0].set_ylabel('# neurons')
fig.suptitle('AdamW: purity histogram evolution', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'purity_evolution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 9 — Crystallization vs grok timing + summary
final_n = n_clean[-1]
crystallize_step = None
for i, n in enumerate(n_clean):
    if n >= 0.8 * final_n: crystallize_step = step_num[i]; break

grok_step_acc95 = None
for i, a in enumerate(accs):
    if a >= 0.95: grok_step_acc95 = step_num[i]; break

# Also: when does it first reach 50%? (for multi-stage check)
step_50pct = None
for i, a in enumerate(accs):
    if a >= 0.50: step_50pct = step_num[i]; break

# Plateau detection: how long is accuracy stuck within 0.45-0.55?
in_plateau = [(0.45 <= a <= 0.55) for a in accs]
plateau_steps = sum(1 for x in in_plateau if x)

print(f'Test acc >= 50% first at step: {step_50pct}')
print(f'Test acc >= 95% first at step: {grok_step_acc95}')
print(f'Steps spent in 45-55% accuracy band: {plateau_steps}')
print(f'Clean neurons reach 80% of final at step: {crystallize_step}')
print(f'Final: acc={accs[-1]:.3f}, n_clean={n_clean[-1]}, max_pur={max_pur[-1]:.3f}, WU CV={wu_cv[-1]:.3f}')

# Multi-stage signature: was there a real plateau?
if step_50pct is not None and grok_step_acc95 is not None:
    gap = grok_step_acc95 - step_50pct
    if gap > 50 and plateau_steps > 30:
        print(f'\n→ MULTI-STAGE: gap between 50% and 95% is {gap} steps with {plateau_steps} plateau steps.')
    else:
        print(f'\n→ MONOLITHIC: gap between 50% and 95% is only {gap} steps; no significant plateau.')
if grok_step_acc95 is not None and crystallize_step is not None:
    diff = crystallize_step - grok_step_acc95
    if diff < 0:
        print(f'→ Fourier neurons crystallize {-diff} steps BEFORE 95% acc')
    elif diff > 0:
        print(f'→ Fourier neurons crystallize {diff} steps AFTER 95% acc')

summary_out = {
    'config': {'optimizer': 'AdamW', 'lr': 1e-3, 'wd': 1.0, 'seed': SEED,
                'steps_recorded': len(results), 'P': P, 'PM1': PM1},
    'step_50pct': int(step_50pct) if step_50pct is not None else None,
    'grok_step_acc95': int(grok_step_acc95) if grok_step_acc95 is not None else None,
    'plateau_steps_45_55': int(plateau_steps),
    'crystallize_step_80pct': int(crystallize_step) if crystallize_step is not None else None,
    'final_n_clean': int(n_clean[-1]), 'final_n_very_clean': int(n_very_clean[-1]),
    'final_max_purity': float(max_pur[-1]), 'final_wu_cv': float(wu_cv[-1]),
    'final_test_acc': float(accs[-1]),
    'per_step': [{
        'step': r['step'], 'test_acc': float(r['test_acc']),
        'n_clean': int(r['n_clean']), 'n_very_clean': int(r['n_very_clean']),
        'max_purity': float(r['max_purity']), 'wu_best_cv': float(r['wu_best_cv']),
        'wu_top_freqs': r['wu_top_freqs'],
    } for r in results],
}
with open(OUT_DIR / 'summary.json', 'w') as f:
    json.dump(summary_out, f, indent=2)
print(f'\nSaved {OUT_DIR / "summary.json"}')

## How to read AdamW slow-mo

**`formation_combined.png` is the headline.** Two y-axes overlaid:
- *Left axis (orange):* test accuracy. Green dashed = 50% marker. Grey dashed = 95% marker.
- *Right axis (blue):* number of clean FFN1 Fourier neurons.

Comparison to Plan 1 Muon:
- Muon: 50% plateau, then 50→99% transition. Multi-stage clear.
- AdamW: ?

**The multi-stage signature** printed in Cell 9 tells us directly whether AdamW shows the same pattern (gap > 50 steps and plateau_steps > 30) or a monolithic single grok.

**`frequency_emergence.png`** shows which Fourier frequencies form when. Compare to Plan 1: did Nyquist k=48 dominate the early phase here too?

## Verdict logic

Three outcomes:
1. **AdamW shows the same multi-stage pattern** (50% plateau + Nyquist first) → multi-stage grokking is real, optimizer-independent. Strong finding for paper.
2. **AdamW shows monolithic single grok** (50% and 95% within ~50 steps) → multi-stage was Muon-specific. Adjust claim: "Muon induces multi-stage emergence; AdamW shows monolithic transition."
3. **Different multi-stage pattern** (e.g. different plateau level, different leading frequency) → understand before claiming.